In [1]:
import ast
import os
import pandas as pd

def extract_genus_name(x):
    try:
        genus_dict = ast.literal_eval(x)
        return genus_dict.get('name', None)
    except:
        return None

os.chdir('/active-data/analysis_results/chr_pla')
all_data = pd.read_csv('genome_chr-pla_statistics.csv')
all_data['genus_clean'] = all_data['genus'].apply(extract_genus_name)
counts = all_data['genus_clean'].value_counts()
keep_genus = counts[counts >= 400].index.to_list()
print(keep_genus)

['Escherichia', 'Klebsiella', 'Staphylococcus', 'Pseudomonas', 'Bacillus', 'Salmonella', 'Streptococcus', 'Streptomyces', 'Acinetobacter', 'Enterococcus', 'Bordetella', 'Enterobacter', 'Xanthomonas', 'Campylobacter', 'Vibrio', 'Mycobacterium', 'Corynebacterium', 'Burkholderia', 'Listeria', 'Citrobacter', 'Helicobacter']


In [2]:
import os
from tqdm import tqdm
import re

def extract_cogs(og_str):
    if pd.isna(og_str) or og_str == "-":
        return set()
    cog_list = re.findall(r"COG\d+", og_str)
    return set(cog_list)

def cog_set(df):
    df["COG_ids"] = df["eggNOG_OGs"].apply(extract_cogs)
    
    all_cogs = set()
    for cogs in df["COG_ids"]:
        all_cogs.update(cogs)
    
    return all_cogs

head = 'query	seed_ortholog	evalue	score	eggNOG_OGs	max_annot_lvl	COG_category	Description	Preferred_name	GOs	EC	KEGG_ko	KEGG_Pathway	KEGG_Module	KEGG_Reaction	KEGG_rclass	BRITE	KEGG_TC	CAZy	BiGG_Reaction	PFAMs'.split('\t')
label = 'pident_90'

for genus_name in keep_genus:
    base_folder = f'/active-data/analysis_results/chr_pla/genus'
    folder = f'{base_folder}/statistics_records/{genus_name}'
    
    os.chdir(folder)
    prediction_re = pd.read_csv('replicon-plasmid_fraction-self_bitscore_statistics.csv')
    
    cog_pd = prediction_re[['accession', 'size', f'category-{label}']].copy()
    with tqdm(total = len(cog_pd), desc=f'{genus_name}({len(cog_pd)})', leave=True, ncols=100, unit='B', unit_scale=True) as pbar:
        for index, row in cog_pd.iterrows():
            contig = row['accession']
            acc = contig.split('-')[0]
            eggnog_result = pd.read_csv(f'/active-data/genomes/bacteria_complete_annotationRefSeq-20250807/eggnog_results/{acc}/{acc}.emapper.annotations', comment = '#', sep = '\t', header = None, names = head)
            eggnog_result = eggnog_result[eggnog_result['query'].str.contains(contig)]
            cogs_set = cog_set(eggnog_result)
            cog_pd.loc[index, 'cog_set'] = str(cogs_set)
            pbar.update(1)
    
    os.chdir(folder)
    cog_pd.to_csv('replicon_cog_set.tsv', sep='\t', index=False)

Escherichia(15436): 100%|███████████████████████████████████████| 15.4k/15.4k [10:58<00:00, 23.4B/s]
Klebsiella(14975): 100%|████████████████████████████████████████| 15.0k/15.0k [10:26<00:00, 23.9B/s]
Staphylococcus(5078): 100%|█████████████████████████████████████| 5.08k/5.08k [01:48<00:00, 46.7B/s]
Pseudomonas(3141): 100%|████████████████████████████████████████| 3.14k/3.14k [02:31<00:00, 20.8B/s]
Bacillus(3992): 100%|███████████████████████████████████████████| 3.99k/3.99k [02:15<00:00, 29.5B/s]
Salmonella(4324): 100%|█████████████████████████████████████████| 4.32k/4.32k [02:53<00:00, 24.9B/s]
Streptococcus(1777): 100%|██████████████████████████████████████| 1.78k/1.78k [00:31<00:00, 56.6B/s]
Streptomyces(2461): 100%|███████████████████████████████████████| 2.46k/2.46k [01:53<00:00, 21.6B/s]
Acinetobacter(3743): 100%|██████████████████████████████████████| 3.74k/3.74k [01:47<00:00, 34.9B/s]
Enterococcus(3287): 100%|███████████████████████████████████████| 3.29k/3.29k [01:05<00:00,